In [1]:
import sys
import os

# Add paths to import from long_form_factuality
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
lff_root = os.path.join(project_root, "long_form_factuality")
for path in [project_root, lff_root]:
    if path not in sys.path:
        sys.path.insert(0, path)

from vllm_wrapper import VLLMAtomizationModel

# Import the original SAFE implementation
from third_party.factscore import atomic_facts
import itertools

def get_atomic_facts_safe(response: str, model, debug=False, query=None):
    """Wrapper that uses the original SAFE implementation with correct paths."""
    demon_dir = os.path.join(lff_root, "third_party", "factscore", "demos")
    atomic_fact_generator = atomic_facts.AtomicFactGenerator(
        api_key='', 
        demon_dir=demon_dir,
        gpt3_cache_file='', 
        other_lm=model,
        query=query
    )
    
    # Monkey patch the generate method to print prompts if debug=True
    if debug:
        original_generate = model.generate
        def debug_generate(prompt, **kwargs):
            print("="*80)
            print("PROMPT SENT TO MODEL:")
            print("="*80)
            print(prompt)
            print("="*80)
            result = original_generate(prompt, **kwargs)
            print("\nMODEL RESPONSE:")
            print("="*80)
            print(result)
            print("="*80)
            return result
        model.generate = debug_generate
    
    facts, _ = atomic_fact_generator.run(response)
    
    # Restore original generate if we patched it
    if debug:
        model.generate = original_generate
    
    # Convert to dict format
    facts_as_dict = [
        {'sentence': sentence, 'atomic_facts': identified_atomic_facts}
        for sentence, identified_atomic_facts in facts
    ]
    all_atomic_facts_list = list(
        itertools.chain.from_iterable([f['atomic_facts'] for f in facts_as_dict])
    )
    
    return {
        'num_claims': len(all_atomic_facts_list),
        'sentences_and_atomic_facts': facts,
        'all_atomic_facts': facts_as_dict,
    }


In [4]:
dummy_text_to_atomize = "He had an IQ of 160"
llm = VLLMAtomizationModel()
atomized = get_atomic_facts_safe(dummy_text_to_atomize, llm, debug=False, query="Tell me about Albert Einstein")

print(atomized)

{'num_claims': 1, 'sentences_and_atomic_facts': [('He had an IQ of 160', ['Albert Einstein had an IQ of 160.'])], 'all_atomic_facts': [{'sentence': 'He had an IQ of 160', 'atomic_facts': ['Albert Einstein had an IQ of 160.']}]}


In [5]:
import json
import os

responses = []
with open(os.getcwd() + "/data_for_git/responses.jsonl", "r") as f:
    for line in f:
        responses.append(json.loads(line))

# print one of the responses

def get_original_prompt(prompt):
    return prompt.split("Based on the documents above, i now want you to: ")[1].split(",")[0]

In [6]:
import threading
from tqdm import tqdm

max_concurrent_requests = 500

request_semaphore = threading.Semaphore(max_concurrent_requests)

# Thread-safe list to collect results
results = []
results_lock = threading.Lock()
error_log = []
error_log_lock = threading.Lock()
def worker(prompt, response, model, pbar):
    try:
        request_semaphore.acquire()
        original_prompt = get_original_prompt(prompt)
        """Worker function that collects results"""
        result = get_atomic_facts_safe(response["response"], model, debug=False, query=original_prompt)
        # print("original prompt", original_prompt)
        # print("prompt", prompt)
        with results_lock:
            results.append({"id": response["id"], "prompt": original_prompt, "response": response["response"], "results": result})
            pbar.update(1)
    except Exception as e:
        print(f"Error processing response {response['id']}: {e}")
        with error_log_lock:
            error_log.append(f"Error processing response {response['id']}: {e}")
    finally:
        request_semaphore.release()

first_n = None

total_tasks = sum(len(q["responses"]) for q in responses[:first_n])

threads = []
# Fix: enumerate returns (index, item), so iterate directly
with tqdm(total=total_tasks, desc="Proce    ssing responses") as pbar:
    for query in responses[:first_n]:
        for response in query["responses"]:
            t = threading.Thread(target=worker, args=(query["prompt"], response, llm, pbar))
            t.start()
            threads.append(t)

    for t in threads:
        t.join()

with open(os.getcwd() + "/data_for_git/atomic_facts_error_log.txt", "w") as f:
    for error in error_log:
        f.write(error + "\n")

# Now results contains all the atomic facts
print(f"Processed {len(results)} responses")
# Access results: results[0], results[1], etc.

Proce    ssing responses:  52%|█████▏    | 1772/3415 [23:31<16:08,  1.70it/s]  Exception in thread Thread-20404 (treat_prompt):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/root/IR_project/long_form_factuality/third_party/factscore/atomic_facts.py", line 270, in treat_prompt
    sentences_from_output = text_to_sentences(output)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/IR_project/long_form_factuality/third_party/factscore/atomic_facts.py", line 306, in text_to_sentences
    sent.strip()[:-1] if sent.strip()[-1] == '\n' else sent.strip()
                         ~~~~~~~~~~~~^^^^
IndexError: string index out of range
Proce    ssing responses:  52%|█████▏    | 1773/3415 [23:34<38:14,  1.40s/it]

Error processing response 44a505d6-a169-45d5-b203-fc647aecbfff: 'However, he faced criticism when he scored an own goal during a Premier League match against Southampton on 11 October 2014, which led to an 8–0 away defeat.'


Proce    ssing responses:  67%|██████▋   | 2304/3415 [30:42<16:11,  1.14it/s]  Exception in thread Thread-23748 (treat_prompt):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1073, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1010, in run
    self._target(*self._args, **self._kwargs)
  File "/root/IR_project/long_form_factuality/third_party/factscore/atomic_facts.py", line 270, in treat_prompt
    sentences_from_output = text_to_sentences(output)
                            ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/IR_project/long_form_factuality/third_party/factscore/atomic_facts.py", line 306, in text_to_sentences
    sent.strip()[:-1] if sent.strip()[-1] == '\n' else sent.strip()
                         ~~~~~~~~~~~~^^^^
IndexError: string index out of range


Error processing response 75d97a94-38e9-4fe8-a8b2-d1bc468a399c: 'She attended primary and secondary school at Colegio San Vicente De Paul in Parque Patricios.'


Proce    ssing responses: 100%|█████████▉| 3413/3415 [44:41<00:01,  1.27it/s]

Processed 3413 responses


In [7]:
# check that the number of sentences in the atomization is the same as the number of sentences in logprobs
# print(responses[0]["responses"])
# match result id:s with response id:s
matches = 0
for result in results:
    for query_responses in responses:
        for response in query_responses["responses"]:
            if result['id'] == response['id']:
                if len(result['results']['sentences_and_atomic_facts']) != len(response['logprobs']):
                    print(f"Number of sentences in atomization ({len(result['results']['sentences_and_atomic_facts'])}) does not match number of sentences in logprobs ({len(response['logprobs'])}) for response {result['id']}")
                else:
                    matches += 1
                    # print(f"Number of sentences in atomization ({len(result['results']['sentences_and_atomic_facts'])}) matches number of sentences in logprobs ({len(response['logprobs'])}) for response {result['id']}")
print(f"Matches: {matches}")

Matches: 3413


In [8]:
import json
with open(os.getcwd() + "/data_for_git/atomic_facts.jsonl", "w") as f:
    for result in results:
        f.write(json.dumps(result) + "\n")
